# Accountable Lending — the audit-trail pipeline, step by step

This notebook walks the same pipeline as `demo.py`, one stage per cell, so each artifact can be inspected.
No LLM is used anywhere: extraction is local spaCy, reasoning is a forward-chained rule.


In [ ]:
import os
os.environ.setdefault('SEMANTICA_DISABLE_PROGRESS', '1')

from demo import (
    stage_ingest, stage_extract, stage_graph, stage_reason,
    stage_decide, stage_audit, stage_export,
)


## Stage 1 — Ingest
Three applicant documents are read through Semantica's `FileIngestor`.


In [ ]:
documents = stage_ingest()
print(f'{len(documents)} documents ingested')


## Stage 2 — Extract
Local spaCy-backed extractors pull entities and relationships out of the documents.


In [ ]:
build_result = stage_extract(documents)


## Stage 3 — Graph
The extractions become a `ContextGraph` — the persistent working memory of the pipeline.


In [ ]:
graph = stage_graph(build_result)


## Stage 4 — Reason
One business rule, forward-chained: `HighRiskFlag(X) AND ThinCreditHistory(X) => RequiresManualReview(X)`.


In [ ]:
conclusions = stage_reason(graph)
conclusions


## Stage 5 — Decide
Three decisions, each recorded as a graph node, with explicit `CAUSED` edges between them.


In [ ]:
decisions = stage_decide(graph)


## Stage 6 — Audit
The regulator's question: *why this outcome?* The graph answers with the causal chain.


In [ ]:
stage_audit(graph, decisions)


## Stage 7 — Export & validate
The graph is exported as RDF/Turtle and checked against SHACL shapes.


In [ ]:
stage_export(graph)


## Takeaway
The documents, the entities, the rule, the decisions, and the causal edges are one connected graph — an audit trail, not a black box.
